# 14.2 - Single-Tool Agent

Status: VERIFIED

## What Are We Solving?
A single-tool agent wraps an LLM in a loop that repeatedly calls one tool until a task is complete or a stop condition is met. This is the fundamental agent architecture.

## Mental Model
Observe -> Decide -> Act -> Repeat. The loop stops when the LLM returns text (no tool call) or the step limit is reached.

In [1]:
import os
import json
from groq import Groq
from dotenv import load_dotenv

load_dotenv()  # loads from .env in project root
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
MODEL = "qwen/qwen3.8-27b"

# Quick test
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Say 'groq ok' only"}],
    max_tokens=10
)
print(f"Groq connected: {r.choices[0].message.content.strip()}")
print(f"Model: {MODEL}")

Groq connected: groq ok
Model: qwen/qwen3.8-27b


## Knowledge Base Search Agent

In [2]:
# Simple knowledge base
KNOWLEDGE_BASE = {
    "python": "Python is a high-level programming language created by Guido van Rossum in 1991.",
    "agent": "An AI agent is an LLM in a controlled loop with tools that can take actions.",
    "rag": "RAG (Retrieval-Augmented Generation) retrieves relevant documents before generating answers.",
    "transformer": "Transformers use self-attention mechanism to process sequential data in parallel.",
    "gradient": "Gradient descent is an optimization algorithm that minimizes loss by updating parameters.",
}

def search_knowledge_base(query: str) -> str:
    query_lower = query.lower()
    results = []
    for key, value in KNOWLEDGE_BASE.items():
        if key in query_lower or any(w in query_lower for w in key.split()):
            results.append(value)
    return json.dumps(results if results else ["No relevant information found."])

tools = [{
    "type": "function",
    "function": {
        "name": "search_knowledge_base",
        "description": "Search internal knowledge base for information about ML, Python, or AI concepts",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"}
            },
            "required": ["query"]
        }
    }
}]

print("Knowledge base loaded:", list(KNOWLEDGE_BASE.keys()))

Knowledge base loaded: ['python', 'agent', 'rag', 'transformer', 'gradient']


## The Agent Loop

In [3]:
def run_agent(user_question: str, max_steps: int = 5) -> dict:
    """Run a single-tool agent with step limiting."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Use the search tool to find information. "
            "When you have enough information, provide a final answer without calling tools."
        )},
        {"role": "user", "content": user_question}
    ]
    
    trace = []
    
    for step in range(max_steps):
        response = client.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        msg = response.choices[0].message
        
        if msg.tool_calls:
            # Append assistant message
            messages.append(msg)
            
            for tc in msg.tool_calls:
                args = json.loads(tc.function.arguments)
                result = search_knowledge_base(**args)
                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": result
                })
                trace.append({"step": step + 1, "tool": tc.function.name, "args": args})
                print(f"  Step {step + 1}: Called {tc.function.name}({args})")
        else:
            trace.append({"step": step + 1, "final_answer": msg.content[:100]})
            print(f"  Step {step + 1}: Final answer")
            return {"answer": msg.content, "steps": step + 1, "trace": trace}
    
    return {"answer": "Max steps reached.", "steps": max_steps, "trace": trace}

# Test
result = run_agent("What is RAG?")
print(f"\nAnswer: {result['answer'][:200]}")
print(f"Steps used: {result['steps']}")

  Step 1: Called search_knowledge_base({'query': 'What is RAG?'})


  Step 2: Final answer

Answer: **RAG (Retrieval-Augmented Generation)** is a technique that retrieves relevant documents (or other information) from a knowledge source before generating an answer. This allows language models to gro
Steps used: 2


## Agent with Multiple Queries

In [4]:
queries = [
    "What is Python?",
    "Explain gradient descent",
    "What are transformers?",
]

for q in queries:
    print(f"\nQuery: {q}")
    r = run_agent(q)
    print(f"  -> {r['answer'][:100]}...")
    print(f"  Steps: {r['steps']}")


Query: What is Python?


  Step 1: Called search_knowledge_base({'query': 'Python programming language overview'})


  Step 2: Final answer
  -> Python is a high-level, general-purpose programming language created by Guido van Rossum and first r...
  Steps: 2

Query: Explain gradient descent


  Step 1: Final answer
  -> I'll use a standard, well-known explanation of gradient descent, as it's a core optimization algorit...
  Steps: 1

Query: What are transformers?


  Step 1: Called search_knowledge_base({'query': 'transformers'})


  Step 2: Final answer
  -> Transformers are a deep learning architecture introduced in 2017 by Google researchers (Vaswani et a...
  Steps: 2


In [5]:
# Verification
assert result["steps"] > 0, "Agent must take at least 1 step"
assert len(result["trace"]) > 0, "Must have trace"
print("VERIFICATION PASSED: Phase 14.2 complete")

VERIFICATION PASSED: Phase 14.2 complete
